In [5]:
import requests, json, time
from pathlib import Path

Soilgrids

In [6]:

ZONES = {
    "ecija":  (37.55537, -5.14244),
    "ubeda":  (38.00720, -3.26722),
    "guadix": (37.24278, -3.16138),
    "jerez":  (36.74074, -6.18370),
}

PROPS = ["clay", "sand", "silt", "soc", "phh2o", "bdod"]
DEPTHS = ["0-5cm", "5-15cm", "15-30cm"]
URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"

def fetch_soilgrids(lat, lon):
    params = {"lat": lat, "lon": lon, "property": PROPS,
              "depth": DEPTHS, "value": "mean"}
    r = requests.get(URL, params=params, timeout=120)
    r.raise_for_status()
    return r.json()

test = fetch_soilgrids(*ZONES["ecija"])
print(list(test.keys()))
print(json.dumps(test["properties"]["layers"][0], indent=1)[:1500])

['type', 'geometry', 'properties', 'query_time_s']
{
 "name": "bdod",
 "unit_measure": {
  "d_factor": 100,
  "mapped_units": "cg/cm\u00b3",
  "target_units": "kg/dm\u00b3",
  "uncertainty_unit": ""
 },
 "depths": [
  {
   "range": {
    "top_depth": 0,
    "bottom_depth": 5,
    "unit_depth": "cm"
   },
   "label": "0-5cm",
   "values": {
    "mean": 144
   }
  },
  {
   "range": {
    "top_depth": 5,
    "bottom_depth": 15,
    "unit_depth": "cm"
   },
   "label": "5-15cm",
   "values": {
    "mean": 156
   }
  },
  {
   "range": {
    "top_depth": 15,
    "bottom_depth": 30,
    "unit_depth": "cm"
   },
   "label": "15-30cm",
   "values": {
    "mean": 159
   }
  }
 ]
}


In [7]:
RAW = Path("../data/raw/soilgrids"); RAW.mkdir(parents=True, exist_ok=True)

for zone, (lat, lon) in ZONES.items():
    out = RAW / f"{zone}_soilgrids.json"
    if out.exists():
        continue                    
    out.write_text(json.dumps(fetch_soilgrids(lat, lon)))
    print("saved", zone)
    time.sleep(5)                    

saved ecija
saved ubeda
saved guadix
saved jerez


In [8]:
import json
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/soilgrids")
OUT = Path("../data/processed/soilgrids")
OUT.mkdir(parents=True, exist_ok=True)

for zone in ZONES:
    data = json.loads((RAW / f"{zone}_soilgrids.json").read_text())

    row = {}
    for layer in data["properties"]["layers"]:
        d_factor = layer["unit_measure"]["d_factor"]
        for depth in layer["depths"]:
            val = depth["values"].get("mean")
            row[f"{layer['name']}_{depth['label']}"] = None if val is None else val / d_factor

    zone_soil = pd.DataFrame([row])          # one row, one column per property and depth
    zone_soil.to_csv(OUT / f"{zone}_soilgrids_static.csv", index=False)
    print("saved", zone)

saved ecija
saved ubeda
saved guadix
saved jerez
